# 05 — RAG with rag_lib

In notebook 04 you saw that LLMs don't remember across turns — you have to decide what's worth keeping. There's a related problem: LLMs don't *know* your documents. They have a snapshot of public text up to some training cutoff. Anything proprietary, anything recent, anything that lives in your filesystem — invisible.

The naive fix is to paste the document into the prompt. That falls apart for three reasons:

- **Cost.** Every token in the prompt is a token you pay for, every turn.
- **Context limits.** A 128K window sounds large until you try to fit a 400-page manual.
- **Lost in the middle.** Models attend unevenly across long contexts. Critical facts buried at position 50,000 often get ignored.

The real fix is **retrieval**: find the few chunks that are actually relevant to the question, give the model just those, and let it answer. That's what RAG (Retrieval-Augmented Generation) is. `rag_lib` is this course's retrieval package.

## The pipeline in four stages

```
ingest    →    retrieve    →    assemble    →    generate
(once)        (per query)      (per query)      (per query)
```

1. **Ingest.** Split documents into chunks, embed them, store in a vector database. Done once per document.
2. **Retrieve.** Embed the query, find the chunks closest to it, return a ranked list.
3. **Assemble.** Pack the top chunks into a prompt along with the question.
4. **Generate.** Send the prompt to an LLM and read the answer.

`rag_lib` wraps all four behind a single class, `RAGPipeline`, and gives you an `inspect_query()` method that opens every stage so you can see *why* it answered the way it did. That inspectability is the point of this notebook.


## Before you run this notebook

You need two Ollama models pulled and the daemon running:

```bash
ollama pull qwen3:8b              # generator
ollama pull nomic-embed-text-v2-moe   # embedder used by rag_lib by default
ollama serve &                    # if it isn't already running as a service
```

`qwen3:8b` is the generator we'll use throughout. It's small enough to be responsive on modest hardware and capable enough to ground answers in retrieved context. `nomic-embed-text-v2-moe` is the embedder `rag_lib` ships with — multilingual, 8K context, friendly to a 24GB GPU.

If you want to follow along on a different model, change the `model` string in the engine cell. Anything Ollama can serve will work; the retrieval logic doesn't care which LLM generates the final answer.


## What's in rag_lib?

Same discovery move as in notebook 04 — see what the package exports before reaching for any particular call.


In [ ]:
import rag_lib

exports = [name for name in dir(rag_lib) if not name.startswith("_")]
print("Exports:", exports)


The main entry point is `RAGPipeline`. `RetrievalTrace` is the data class returned by `inspect_query()`. `IngestResult` is what `ingest()` gives you back. The rest are exceptions.

`describe_rag_pipeline` is the interop hook that lets `llm_inspector_ui` introspect a pipeline — useful later, not needed now.


## A small corpus to play with

Real RAG systems index thousands of documents. For teaching, we'll use four short notes. Two of them deliberately share a keyword — "Mercury" — to set up the failure mode we'll diagnose at the end.


In [ ]:
import tempfile
from pathlib import Path

corpus_dir = Path(tempfile.mkdtemp(prefix="rag_nb05_"))
print(f"Corpus directory: {corpus_dir}")

docs = {
    "mercury_planet.md": """# Mercury (planet)

Mercury is the smallest and innermost planet in the Solar System. Its
orbital period around the Sun is 88 Earth days, the shortest of any
planet. Mercury has no atmosphere to retain heat, so surface temperatures
swing from about 430°C in daylight to -180°C at night.
""",
    "project_mercury_ops.md": """# Project Mercury — Operations Note

Project Mercury is our customer telemetry pipeline. The production
deployment runs in the us-west-2 region. Ingest endpoints are behind a
load balancer at telemetry.example.internal. Average daily volume is
2.4 billion events. On-call rotation is documented in the runbook.
""",
    "embeddings_intro.md": """# What an Embedding Is

An embedding is a vector — a list of numbers — that represents a piece
of text in a way that captures meaning. Texts with similar meaning have
embeddings that are close together in vector space. Retrieval works by
embedding the query, then finding stored embeddings nearest to it,
typically by cosine similarity.
""",
    "chunking_basics.md": """# Why Chunking Matters

A document is usually too long to embed as one vector. You split it
into chunks first. The chunk size is a tradeoff: smaller chunks give
sharper relevance but lose context; larger chunks preserve context but
dilute the signal. rag_lib supports several chunking strategies and
picks one based on the doc_type you pass to ingest().
""",
}

for name, text in docs.items():
    (corpus_dir / name).write_text(text)

print(f"Wrote {len(docs)} files.")


## Build a pipeline

`RAGPipeline()` with no arguments reads `~/.rag_lib/rag_lib.yaml` (auto-created on first use). For this notebook we'll pass an explicit config dict so the ChromaDB store lands in a scratch directory and doesn't clobber anything on your machine.


In [ ]:
from rag_lib import RAGPipeline

scratch = Path(tempfile.mkdtemp(prefix="rag_nb05_store_"))

config = {
    "embedder": {
        "host": "http://localhost:11434",
        "model": "nomic-embed-text-v2-moe",
        "timeout": 120,
    },
    "storage": {
        "backend": "chromadb",
        "path": str(scratch / "chroma"),
        "collection_prefix": "rag_nb05_",
        "bm25_path": str(scratch / "bm25"),
    },
    "chunker": {
        "max_embed_tokens": 1800,
        "defaults": {"strategy": "fixed_size", "chunk_size": 512, "chunk_overlap": 50},
        "doc_types": {
            "guide": {"strategy": "sentence_window", "window_size": 3},
        },
    },
    "retriever": {
        "n_results": 5,
        "max_context_tokens": 2000,
    },
}

pipeline = RAGPipeline(config=config)
print("Pipeline ready.")
print("Components:", pipeline.describe_component().features)


`describe_component()` returns a `CapabilityDescriptor` — the same shape every package in the suite uses to advertise what it can do. The `features` tuple tells you which capabilities are wired in. For a default pipeline you'll see hybrid retrieval and trace events. Reranking and query expansion are off in this config; they're notebook 06 territory.


## Ingest the corpus

Pass each file to `pipeline.ingest()`. The `doc_type` argument tells the chunker which strategy to use. For these short prose notes, "guide" — which uses sentence-window chunking — is appropriate.


In [ ]:
results = []
for path in sorted(corpus_dir.glob("*.md")):
    result = pipeline.ingest(path, doc_type="guide")
    results.append(result)
    print(f"{path.name:30s}  chunks_stored={result.chunks_stored}  ok={result.success}")


Each call to `ingest()` does three things:

1. **Load** the file (the loader auto-detects format — markdown, PDF, HTML, etc.).
2. **Chunk** it into pieces under the embedder's token limit, using the strategy for the given `doc_type`.
3. **Embed and store** each chunk in ChromaDB and update the BM25 index.

For four small files this finishes in a few seconds. For a thousand files it can take minutes — which is why `rag_lib` keeps the embedder model loaded between calls (the `keep_alive: 600` setting in the embedder config).


## Retrieve

Now the interesting part. Given a question, `retrieve()` returns a ranked list of chunks. Each chunk has the source document, the chunk text, and a similarity score.


In [ ]:
chunks = pipeline.retrieve("What is an embedding?")

for i, chunk in enumerate(chunks, 1):
    print(f"[{i}] score={chunk.score:.3f}  source={chunk.source_id}")
    print(f"    {chunk.content[:120]}...")
    print()


Two things to notice:

- **The order matters.** The top result should be from `embeddings_intro.md`. If it isn't, something is wrong with embedding, storage, or scoring — not the LLM.
- **The score is comparable, not absolute.** Cosine similarity ranges from -1 to 1; in practice retrieved chunks land in the 0.3–0.8 range. A score of 0.5 doesn't mean "50% relevant" — it means "this beat the alternatives by some margin."

This is the most useful debugging instinct RAG can teach you: **when the model gives a bad answer, look at retrieval first, not the model.** The model can only answer with what it's given.


## Assemble the prompt

`assemble_prompt()` packs the top chunks into a context block, prepends them to the user's question, and respects a token budget. Same idea as the token budget from notebook 04 — the budget belongs to *you*, and the assembler keeps you under it.


In [ ]:
prompt = pipeline.assemble_prompt(
    query="What is an embedding?",
    chunks=chunks,
    max_context_tokens=1500,
)
print(prompt)


The exact format is rag_lib's default template. You can override it; for now, notice the structure:

- Each chunk is labeled with its source so the model can cite.
- The question comes last, which most models attend to better than question-first layouts.
- There's no preamble telling the model to "use the context" — that goes in the system prompt, which is the next cell.


## Generate

Bring in `llm_engines` to run the prompt through `qwen3:8b`.


In [ ]:
from llm_engines import get_engine
from llm_engines.contracts import GenerationRequest

engine = get_engine("ollama", "qwen3:8b")

system_prompt = (
    "Answer the question using only the provided context. "
    "If the context does not contain the answer, say so. "
    "Cite the source filename when you can."
)

request = GenerationRequest(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ],
    temperature=0.2,
)

response = engine.generate(request)
print(response.message.content)


A few choices worth flagging:

- **Low temperature (0.2).** RAG answers should be grounded, not creative. Sampling noise is the enemy of "did it use the context."
- **Explicit "say so if missing" in the system prompt.** Without this, models prefer to guess. With it, you get honest "the context doesn't say" answers that are easier to trust.
- **Source attribution in the system prompt.** Doesn't guarantee citations, but most modern instruction-tuned models will comply, and it makes failures easier to spot.


## Aside: I/O vs reasoning

You just ran two models in this pipeline: an embedder (`nomic-embed-text-v2-moe`) and a generator (`qwen3:8b`). That split is the simplest example of a broader cost-architecture pattern.

The embedder is doing **I/O work** — it reads text in, produces vectors out, and the quality bar is "does the vector capture meaning." It doesn't need to reason. A small specialized model is exactly right for the job.

The generator is doing **reasoning work** — it has to read the retrieved chunks, understand the question, and synthesize an answer. The quality bar is much higher. A small model can do this on simple queries; complex ones benefit from a bigger model.

That split scales up. For students working on GPU-limited hardware, the same pattern lets you run a small local model for bulk I/O tasks (summarizing files, generating boilerplate, building chunk summaries) and reserve calls to a larger model — local or API — for tasks that genuinely need reasoning. The retrieval layer doesn't have to be a vector database, either: any tool that surfaces relevant context cheaply (file search, SQL, graph traversal) plays the same role. Notebook 06 expands on this with hybrid retrieval and reranking, and a later module covers the routing pattern explicitly.

The general lesson: when you find yourself burning expensive tokens on tasks that don't need judgment, that's a signal to route those tasks elsewhere.


## Make it inspectable

Everything above hides the stages behind `retrieve()`. When something goes wrong — wrong chunk picked, irrelevant doc returned, model hallucinates — you need to see what each stage actually did.

`inspect_query()` returns a `RetrievalTrace` with every stage's output.


In [ ]:
trace = pipeline.inspect_query("What is an embedding?")

print("Query variants:", trace.query_variants)
print()
print(f"Dense retrieval returned {len(trace.dense_results)} candidates:")
for doc in trace.dense_results[:3]:
    print(f"  score={doc.score:.3f}  source={doc.source}")
print()
print(f"BM25 retrieval returned {len(trace.bm25_results)} candidates:")
for doc in trace.bm25_results[:3]:
    print(f"  score={doc.score:.3f}  source={doc.source}")
print()
print(f"Fused (hybrid) results: {len(trace.fused_results)}")
print(f"Selected for prompt: {len(trace.selected_results)}")


The trace exposes every stage:

- `dense_results` — semantic similarity over embeddings.
- `bm25_results` — keyword/lexical scoring.
- `fused_results` — both lists merged by reciprocal rank fusion (RRF).
- `reranked_results` — empty here because reranking is off in this config.
- `selected_results` — what actually ended up in the prompt.
- `events` — a structured log of what happened, suitable for `llm_inspector` analysis.

The reason `rag_lib` does both dense and BM25 by default is that they fail in different ways. Dense retrieval is great at paraphrase and synonyms but can miss rare proper nouns. BM25 is great at exact-keyword matches but blind to meaning. Combining them with RRF gets you the best of both — most of the time.


## A failure mode you can see

Here's a query that exposes how retrieval can pick the wrong chunk on a keyword collision.


In [ ]:
bad_query = "What region does Project Mercury run in?"
trace = pipeline.inspect_query(bad_query)

print(f"Query: {bad_query}")
print()
print("Top dense candidates:")
for doc in trace.dense_results[:3]:
    print(f"  score={doc.score:.3f}  source={doc.source}")
    print(f"    {doc.text[:100]}...")
    print()


On a small corpus like this the dense retriever usually gets it right, but you can see the planet Mercury note sitting in the candidate list — close enough on "Mercury" to compete. On a real corpus with hundreds of mentions of "Mercury" in unrelated contexts, the top result can flip to the wrong document, and the LLM will then dutifully tell you that Project Mercury orbits the Sun.

This is the canonical broken-RAG failure: **the retriever returned text that's similar to the query, not relevant to the question.** Similarity is not relevance.

Run the same query through the full pipeline and watch what happens:


In [ ]:
chunks = pipeline.retrieve(bad_query)
prompt = pipeline.assemble_prompt(bad_query, chunks, max_context_tokens=1500)

request = GenerationRequest(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ],
    temperature=0.2,
)
response = engine.generate(request)
print(response.message.content)


If the top retrieved chunk was the operations note, the model should correctly say `us-west-2`. If the planet note ranked first, the model might say "Project Mercury is a planet" or refuse to answer — both are honest, neither is useful.

Either way, **you can tell which from the trace, not from the answer alone.** That's the whole point of making retrieval inspectable.

The fix for this failure mode is one of notebook 06's topics: lexical guardrails, doc-type filtering, and cross-encoder reranking that scores `(query, chunk)` jointly rather than independently. The repo's `docs/tutorials/broken_rag_lab.md` walks through the full repair with code; this notebook is just here to make the failure visible.


## Learning checkpoint

Before moving on, you should be able to answer these without re-running anything:

1. What are the four stages of a RAG pipeline, and which run once vs per query?
2. Why does `rag_lib` combine dense and BM25 retrieval by default? When does each one fail?
3. If the model gives a wrong answer, what's the first thing you should look at — the prompt, the retrieval, or the model output?
4. Why is similarity not the same as relevance? Give an example.
5. What does `inspect_query()` give you that `retrieve()` doesn't?

## What's next

- **Notebook 06** picks this up: hybrid weighting tuning, cross-encoder reranking, HyDE query expansion, and RAGAS evaluation. It assumes the mental model you just built.
- **`course/starter_projects/source_grounded_qa/`** is a runnable scaffold you can extend with your own documents.
- **`docs/tutorials/broken_rag_lab.md`** walks through the Project Mercury fix end-to-end.

## Clean up

```python
import shutil
shutil.rmtree(corpus_dir, ignore_errors=True)
shutil.rmtree(scratch, ignore_errors=True)
```
